In [1]:
%useLatestDescriptors
%use koog

In [9]:
val agentStrategy = strategy<String, String>("是否调用工具策略图") {
    val nodeSendInput by nodeLLMRequest()
    val nodeExecuteTool by nodeExecuteTool()
    val nodeSendToolResult by nodeLLMSendToolResult()

    edge(nodeStart forwardTo nodeSendInput)

    edge(
        (nodeSendInput forwardTo nodeFinish)
            .transformed { it }
            .onAssistantMessage { true }
    )

    edge(
        (nodeSendInput forwardTo nodeExecuteTool)
            .onToolCall { true }
    )

    edge(nodeExecuteTool forwardTo nodeSendToolResult)

    edge(
        (nodeSendToolResult forwardTo nodeFinish)
            .transformed { it }
            .onAssistantMessage { true }
    )

    edge(
        (nodeSendToolResult forwardTo nodeExecuteTool)
            .onToolCall { true }
    )
}

In [10]:
//    val mermaidDiagram: String = agentStrategy.asMermaidDiagram()

//    println(mermaidDiagram)

val mraidDiagram : String = agentStrategy.asMermaidDiagram()
println(mraidDiagram)

---
title: 是否调用工具策略图
---
stateDiagram
    state "nodeSendInput" as nodeSendInput
    state "nodeExecuteTool" as nodeExecuteTool
    state "nodeSendToolResult" as nodeSendToolResult

    [*] --> nodeSendInput
    nodeSendInput --> [*] : transformed
    nodeSendInput --> nodeExecuteTool : onCondition
    nodeExecuteTool --> nodeSendToolResult
    nodeSendToolResult --> [*] : transformed
    nodeSendToolResult --> nodeExecuteTool : onCondition


In [ ]:
println("")

In [7]:
import ai.koog.agents.core.environment.ReceivedToolResult

val myStrategy = strategy<String, String>("策略图"){
    val nodeSendInput by nodeLLMRequest() //
    val nodeExecutor by nodeExecuteTool()
    val nodeSendToolResult by nodeLLMSendToolResult()
    val compressHistory by nodeLLMCompressHistory<ReceivedToolResult>()
    edge(nodeStart forwardTo nodeSendInput)
    edge((nodeSendInput forwardTo nodeFinish).transformed{msg-> msg.content}.onAssistantMessage { true })

    edge(
        (nodeSendInput forwardTo nodeExecutor).onToolCall { true }
    )
    edge(
        (nodeExecutor forwardTo compressHistory)
            .onCondition { llm.readSession { prompt.messages.size > 10 } }
    )

    edge(
        (nodeExecutor forwardTo nodeSendToolResult)
    )
    edge(
        (nodeSendToolResult forwardTo nodeFinish).transformed { it }.onAssistantMessage { true }
    )
    edge(
        (nodeSendToolResult forwardTo nodeExecutor).onToolCall { true }
    )
}

val  mraidDiagram : String = myStrategy.asMermaidDiagram()
println(mraidDiagram)


---
title: 策略图
---
stateDiagram
    state "nodeSendInput" as nodeSendInput
    state "nodeExecutor" as nodeExecutor
    state "compressHistory" as compressHistory
    state "nodeSendToolResult" as nodeSendToolResult

    [*] --> nodeSendInput
    nodeSendInput --> [*] : transformed
    nodeSendInput --> nodeExecutor : onCondition
    nodeExecutor --> compressHistory : onCondition
    nodeExecutor --> nodeSendToolResult
    nodeSendToolResult --> [*] : transformed
    nodeSendToolResult --> nodeExecutor : onCondition


In [8]:
// Define that the history is too long if there are more than 100 messages
//private suspend fun AIAgentContext.historyIsTooLong(): Boolean = llm.readSession { prompt.messages.size > 100 }

val strategy = strategy<String, String>("execute-with-history-compression") {
    val callLLM by nodeLLMRequest()
    val executeTool by nodeExecuteTool()
    val sendToolResult by nodeLLMSendToolResult()

    // Compress the LLM history and keep the current ReceivedToolResult for the next node
    val compressHistory by nodeLLMCompressHistory<ReceivedToolResult>()

    edge(nodeStart forwardTo callLLM)
    edge(callLLM forwardTo nodeFinish onAssistantMessage { true })
    edge(callLLM forwardTo executeTool onToolCall { true })

    // Compress history after executing any tool if the history is too long
    edge((executeTool forwardTo compressHistory).onCondition { llm.readSession { prompt.messages.size > 10 } } )
    edge(compressHistory forwardTo sendToolResult)
    // Otherwise, proceed to the next LLM request
    edge(executeTool forwardTo sendToolResult onCondition { !llm.readSession { prompt.messages.size > 10 } })

    edge(sendToolResult forwardTo executeTool onToolCall { true })
    edge(sendToolResult forwardTo nodeFinish onAssistantMessage { true })
}

val  Mystrategy : String = strategy.asMermaidDiagram()
println(Mystrategy)

---
title: execute-with-history-compression
---
stateDiagram
    state "callLLM" as callLLM
    state "executeTool" as executeTool
    state "compressHistory" as compressHistory
    state "sendToolResult" as sendToolResult

    [*] --> callLLM
    callLLM --> [*] : transformed
    callLLM --> executeTool : onCondition
    executeTool --> compressHistory : onCondition
    executeTool --> sendToolResult : onCondition
    compressHistory --> sendToolResult
    sendToolResult --> executeTool : onCondition
    sendToolResult --> [*] : transformed


In [13]:
val agentStrategy = strategy<String, String>("是否调用工具策略图") {
    val nodeSendInput by nodeLLMRequest()
    val nodeExecuteTool by nodeExecuteTool()
    val nodeSendToolResult by nodeLLMSendToolResult()
    val compressHistory by nodeLLMCompressHistory<ReceivedToolResult>()

    edge(nodeStart forwardTo nodeSendInput)

    edge(
        (nodeSendInput forwardTo nodeFinish)
            .transformed { it }
            .onAssistantMessage { true }
    )

    edge(
        (nodeSendInput forwardTo nodeExecuteTool)
            .onToolCall { true }
    )
    edge(
        (nodeExecuteTool forwardTo compressHistory)
            .onCondition { llm.readSession { prompt.messages.size >10 } }
    )

    edge(
        (nodeExecuteTool forwardTo nodeSendToolResult)
            .onCondition { !llm.readSession { prompt.messages.size >10 } }
    )

    edge(
        (compressHistory forwardTo nodeSendToolResult)

    )

//    edge(nodeExecuteTool forwardTo nodeSendToolResult)

    edge(
        (nodeSendToolResult forwardTo nodeFinish)
            .transformed { it }
            .onAssistantMessage { true }
    )

    edge(
        (nodeSendToolResult forwardTo nodeExecuteTool)
            .onToolCall { true }
    )
}

val  Mystrategy : String = agentStrategy.asMermaidDiagram()
println(Mystrategy)

---
title: 是否调用工具策略图
---
stateDiagram
    state "nodeSendInput" as nodeSendInput
    state "nodeExecuteTool" as nodeExecuteTool
    state "compressHistory" as compressHistory
    state "nodeSendToolResult" as nodeSendToolResult

    [*] --> nodeSendInput
    nodeSendInput --> [*] : transformed
    nodeSendInput --> nodeExecuteTool : onCondition
    nodeExecuteTool --> compressHistory : onCondition
    nodeExecuteTool --> nodeSendToolResult : onCondition
    compressHistory --> nodeSendToolResult
    nodeSendToolResult --> [*] : transformed
    nodeSendToolResult --> nodeExecuteTool : onCondition
